In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Gerando dados sintéticos estruturados
np.random.seed(42)
n_registros = 120

datas = pd.date_range(start='2024-01-01', periods=n_registros, freq='D')
categorias = ['Eletronicos', 'Livros', 'Roupas', 'Automotivo']
estados = ['SP', 'RJ', 'MG', 'RS', 'BA']

dados = {
    'data': datas,
    'categoria': np.random.choice(categorias, size=n_registros),
    'estado': np.random.choice(estados, size=n_registros),
    'valor': np.random.uniform(50, 4500, size=n_registros).round(2),
    'quantidade': np.random.randint(1, 15, size=n_registros),
}

df = pd.DataFrame(dados)
df['valor_venda'] = (df['valor'] * np.random.uniform(0.9, 1.1, size=n_registros)).round(2)

## Parte 1: Gráficos Essenciais e Personalização (Básico)

### 1. Faturamento mensal (gráfico de linha)

In [ ]:
faturamento_mensal = df.set_index('data')['valor_venda'].resample('ME').sum()

plt.figure(figsize=(10, 5))
plt.plot(
    faturamento_mensal.index,
    faturamento_mensal.values,
    marker='o',
    linestyle='--',
    color='#2E86AB',
)
plt.title('Faturamento Mensal Total')
plt.xlabel('Mês')
plt.ylabel('Faturamento (R$)')
plt.grid(True)
plt.tight_layout()
plt.show()

### 2. Total de vendas por categoria (gráfico de barras)

In [ ]:
vendas_por_categoria = df.groupby('categoria')['valor_venda'].sum().sort_values()

plt.figure(figsize=(8, 5))
plt.bar(vendas_por_categoria.index, vendas_por_categoria.values, color='#A23B72')
plt.title('Total de Vendas por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Total Vendido (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3. Distribuição dos valores de venda (histograma)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['valor'], bins=10, edgecolor='black', color='#F18F01')
plt.title('Distribuição dos Valores de Venda')
plt.xlabel('Valor (R$)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

## Parte 2: API Orientada a Objetos e Eixos Múltiplos (Intermediário)

### 4. Subplots 2x1 com eixo X compartilhado (faturamento x quantidade)

In [ ]:
df_mensal = df.set_index('data').resample('ME').agg(
    faturamento=('valor_venda', 'sum'),
    quantidade=('quantidade', 'sum'),
)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 7))

ax1.plot(
    df_mensal.index,
    df_mensal['faturamento'],
    marker='o',
    linestyle='--',
    color='#2E86AB',
)
ax1.set_ylabel('Faturamento (R$)')
ax1.set_title('Faturamento Mensal')
ax1.grid(True)

ax2.bar(df_mensal.index, df_mensal['quantidade'], width=20, color='#A23B72')
ax2.set_ylabel('Quantidade Vendida')
ax2.set_xlabel('Mês')
ax2.set_title('Quantidade de Itens Vendidos por Mês')

plt.tight_layout()
plt.show()

### 5. Eixos duplos (twinx): valor total x quantidade média por venda

In [ ]:
df_mensal_twinx = df.set_index('data').resample('ME').agg(
    valor_total=('valor_venda', 'sum'),
    qtd_media=('quantidade', 'mean'),
)

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.bar(
    df_mensal_twinx.index,
    df_mensal_twinx['valor_total'],
    width=20,
    color='#2E86AB',
    label='Valor Total (R$)',
)
ax1.set_xlabel('Mês')
ax1.set_ylabel('Valor Total Vendido (R$)', color='#2E86AB')
ax1.tick_params(axis='y', labelcolor='#2E86AB')

ax2 = ax1.twinx()
ax2.plot(
    df_mensal_twinx.index,
    df_mensal_twinx['qtd_media'],
    marker='o',
    color='#F18F01',
    label='Qtd Média por Venda',
)
ax2.set_ylabel('Quantidade Média de Itens por Venda', color='#F18F01')
ax2.tick_params(axis='y', labelcolor='#F18F01')

plt.title('Valor Total Vendido x Quantidade Média de Itens por Mês')
fig.tight_layout()
plt.show()

### 6. Anotação destacando o mês de maior faturamento

In [ ]:
faturamento_mensal = df.set_index('data')['valor_venda'].resample('ME').sum()
mes_pico = faturamento_mensal.idxmax()
valor_pico = faturamento_mensal.max()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    faturamento_mensal.index,
    faturamento_mensal.values,
    marker='o',
    linestyle='--',
    color='#2E86AB',
)
ax.set_title('Faturamento Mensal com Destaque para o Pico')
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)')
ax.grid(True)

ax.annotate(
    f'Pico: R$ {valor_pico:,.2f}',
    xy=(mes_pico, valor_pico),
    xytext=(mes_pico, valor_pico * 1.15),
    arrowprops=dict(facecolor='black', arrowstyle='->'),
    ha='center',
)
ax.set_ylim(top=faturamento_mensal.max() * 1.3)

plt.tight_layout()
plt.show()

## Parte 3: Matriz de Intensidade / Heatmap (Intermediário / Avançado)

### 7. Tabela dinâmica: valor total por categoria x estado

In [ ]:
pivot_vendas = pd.pivot_table(
    df,
    values='valor_venda',
    index='categoria',
    columns='estado',
    aggfunc='sum',
    fill_value=0,
)
pivot_vendas

### 8 e 9. Heatmap com imshow, colorbar e exportação em PNG (300 DPI)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot_vendas.values, cmap='viridis', aspect='auto')

ax.set_xticks(range(len(pivot_vendas.columns)))
ax.set_xticklabels(pivot_vendas.columns)
ax.set_yticks(range(len(pivot_vendas.index)))
ax.set_yticklabels(pivot_vendas.index)

ax.set_xlabel('Estado')
ax.set_ylabel('Categoria')
ax.set_title('Total de Vendas por Categoria e Estado')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Total Vendido (R$)')

plt.tight_layout()
plt.savefig('heatmap_vendas.png', dpi=300, bbox_inches='tight')
plt.show()